# 엑셀/PDF 파일 읽기 실습

- 엑셀 파일은 `openpyxl`, PDF 파일은 `pypdf` 패키지를 사용


## 패키지 설치

터미널에서 아래 명령어를 한 번 실행한다.

```bash
python -m pip install openpyxl pypdf
```


## 엑셀 파일 읽기

엑셀 파일은 단순 문자열 파일이 아니라 여러 시트, 셀, 서식 정보가 들어 있는 구조화 파일이다. 그래서 `openpyxl` 같은 전용 라이브러리로 읽는다.


In [ ]:
 # 엑셀을 열어서 파이썬 자체객체인 workbook으로 변환한다
from openpyxl import load_workbook
from openpyxl.chart import reader

# data_only=true : 셀에 작성된 데이터 읽어오기(수식 제외, 결과만 읽어옴)
workbool = load_workbook('students.xlsx', read_only=True)
print(workbool.sheetnames)  # 엑셀의 모든 시트를 읽어옴
work_sheet = workbool['students']
# 시트 이름
print(work_sheet.title)
# 데이터의 몇행 몇열인지 구분
print(f'max_row : {work_sheet.max_row}, max_column: {work_sheet.max_column}')

In [ ]:
# 특정 셀 값 읽어오기
print("B2값 :", work_sheet['B2'].value)
print("D3값 :", work_sheet['D3'].value)

print("4행 3열 값 :", work_sheet.cell(row=4,column=2).value)



In [ ]:
# 엑셀 한 줄씩 일기
# iter == iterator == 반복자
# iter_rows == 행 반복자 : 반복 될 때마다 다음 행 반환
# values_only=True : 데이터만 가져오는데 튜플값으로 가져옴
for row in work_sheet.iter_rows(values_only=True):
    print(row)



# rows = work_sheet.iter_rows(values_only=True)
# headers = next(rows)
# users = []
#
# for row in rows:
#     users.append(dict(zip(headers, row)))
#
# print(users)


In [ ]:
# 엑셀 행 데이터를 dict list로 변환
rows = work_sheet.iter_rows(values_only=True)
print("리스트 전 : ",rows)
lst = list(rows)
print("데이터 처리 용이하게 list 변경 : ", lst)

# key 값을 header로 뽑아옴
headers = lst[0]
student_list:list[dict] = [] # 결과 저장용 변수
for row in lst[1:]:
    # 학생 dict 데이터 생성
    # zip : 두 튜플의 같은 인덱스 요소끼리 k:v 형식으로 묶어서 반환한다
    student = dict(zip(headers, row))
    student_list.append(student)



In [ ]:
# 엑셀에서 읽어온 학생 데이터로 학생수, 점수합계, 평균,최고점,최저점,
# 내꺼

print(f"학생수 : {len(student_list)}")
print(f"점수 합계 : {sum([int(student['score']) for student in student_list])}" )
print(f"점수 평군 : {sum([int(student['score']) for student in student_list]) / len(student_list)}" )
print(f'최고 점수 : {max([int(student['score']) for student in student_list])}, ')
print(f'최저 점수 : {min([int(student['score']) for student in student_list])},')


bottom_student = min(
    student_list,
    key=lambda student: student['score']
)

# 강사님 풀이
scores:list[int] = [student['score'] for student in student_list]

print("학생수 : ", len(scores))
print("점수 합계 : ", sum(scores))
print("점수 평군 : ", sum(scores)/len(scores))
print("점수 최고  : ", max(scores))
print("점수 최저 : ", min(scores))


## PDF 파일 읽기

- PDF는 페이지, 글자 위치, 폰트, 이미지 정보가 함께 들어 있는 문서 파일이다. 그래서 텍스트 파일처럼 바로 읽기보다 `pypdf` 같은 라이브러리로 페이지 단위 텍스트를 추출한다.

- 한글 PDF는 폰트와 인코딩 방식에 따라 텍스트 추출 결과가 달라질 수 있음



In [ ]:
# pypdf: 파이썬에서 pdf 파일을 읽고, 쓸수있게 하는 라이브러리(모듈)
from pypdf import PdfReader

#pathlib.Path : 파일/폴더 경로를 편하게 다루기 위한 클래스
# 현재 스크립트가 실행되는 위치를 기준으로 판단한다
from pathlib import Path
pdf_path = Path('io_sample.pdf')
print(pdf_path)

reader = PdfReader(pdf_path)
print(reader, type(reader))

# pdfReader.pages : 현재 읽어들인 pdf 파일의 페이지 객체를 담아둔 list
print("reader.pages : ", reader.pages)
print("읽은 pdf의 페이지 수 : ", len(reader.pages))



In [11]:
# 첫번째 페이지 텍스트 추출
first_page_text = reader.pages[0].extract_text()
print("첫번째 페이지 텍스트 : ", first_page_text)
print("두번째 페이지 텍스트 : ", reader.pages[1].extract_text())

첫번째 페이지 텍스트 :  IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.



In [19]:
# 전체 페페이지 추출
# 단 페이지별로 구분을 해서
for i, p in enumerate(reader.pages, start=1):
    print(f'{i} 페이지')
    print(f'내용 : \n{p.extract_text()}')

1 페이지
내용 : 
IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.

2 페이지
내용 : 
Second Page
A PDF can contain multiple pages.
In Python, each page can be read and processed separately.

